In [ ]:
%sql
WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      -- Use backticks for spaces, or replace with exact column name if it has underscores
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

CC_Mapped_AUs AS (
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS Mapped_AU_ID
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL
)

SELECT 
    COALESCE(m.BusinessID, c.Mapped_AU_ID) AS BusinessID,
    m.AU_Name, 
    CASE 
        WHEN c.Mapped_AU_ID IS NULL THEN 'Master AU list ONLY'
        WHEN m.BusinessID IS NULL THEN 'CC ONLY'
    END AS Source_Table

FROM Master_AUs m
FULL OUTER JOIN CC_Mapped_AUs c
    ON m.BusinessID = c.Mapped_AU_ID
    
-- This drops the records that exist in both tables
WHERE m.BusinessID IS NULL 
   OR c.Mapped_AU_ID IS NULL
   
ORDER BY BusinessID;